# Unidad 45. EDA reproducible y comunicación

**Bloque:** Seaborn (UT3) · **Tiempo orientativo:** 5–7 horas

> Un análisis exploratorio reproducible conserva el camino desde datos locales hasta una conclusión limitada: inspección, transformación explícita, figura legible y exportación que otra persona pueda repetir.


## Resultados de aprendizaje

Al terminar podrás combinar filtrado, columnas derivadas, agrupaciones y tablas pivote de Pandas con figuras de Seaborn; elegir paletas y etiquetas accesibles; refinar una figura con Matplotlib; y exportarla en una ruta relativa.

**Prerequisitos:** unidades 18–23, 31–35 y 39–44.


In [ ]:
from pathlib import Path
import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns

sns.set_theme(style="whitegrid", context="notebook")

def ruta_datos(nombre):
    for ruta in [Path("datos") / "seaborn" / nombre, Path("..") / "datos" / "seaborn" / nombre]:
        if ruta.exists(): return ruta
    raise FileNotFoundError(f"No se encontró {nombre}. Abre el cuaderno desde la carpeta del curso.")

observaciones = pd.read_csv(ruta_datos("44_observaciones_modelo.csv"))
observaciones.info()


## 1. Un flujo que otra persona pueda revisar

Un EDA no es una colección de gráficos. Empieza con una pregunta, inspecciona las columnas y la calidad, transforma con operaciones visibles y produce figuras que respondan a esa pregunta. Evita modificar el `DataFrame` original sin dejar rastro.

Pregunta de ejemplo: **«¿Qué equilibrio aparece entre exactitud, latencia y energía en las observaciones simuladas?»**. Los datos son sintéticos; la respuesta solo describe esta tabla.


In [ ]:
assert observaciones.isna().sum().sum() == 0

analisis = observaciones.assign(
    eficiencia_pct_wh=observaciones["exactitud_pct"] / observaciones["energia_wh"],
    nivel_latencia=pd.cut(observaciones["latencia_ms"], bins=[0, 150, 200, 300], labels=["baja", "media", "alta"]),
)
analisis.head()


## 2. Resumir con Pandas antes de dibujar

Una agrupación responde a una pregunta de resumen. La siguiente tabla muestra medias por dispositivo y configuración; no sustituye a las observaciones individuales. Una tabla pivote deja visible qué filas, columnas, medida y agregación intervienen. Revisa ambas antes de interpretar barras o colores.


In [ ]:
resumen = analisis.groupby(["dispositivo", "configuracion"], as_index=False).agg(exactitud_media=("exactitud_pct", "mean"), latencia_media=("latencia_ms", "mean"), energia_media=("energia_wh", "mean"))
display(resumen)

pivote = analisis.pivot_table(index="configuracion", columns="dispositivo", values="eficiencia_pct_wh", aggfunc="mean", observed=False)
display(pivote)


## 3. Comunicar sin depender solo del color

Una paleta debe diferenciar grupos, pero el color no debe ser la única señal. Aquí `hue` se refuerza con `style`, las etiquetas incluyen unidades y el título describe exactamente la relación. Matplotlib permite añadir ajustes razonados y exportar la figura.

No recortes el eje vertical para exagerar diferencias pequeñas. Una anotación debe señalar una observación o decisión que ayude a leer, no decorar el gráfico.


In [ ]:
paleta = {"base": "#4477AA", "ajustada": "#CC6677"}
fig, ax = plt.subplots(figsize=(8, 5))
sns.scatterplot(data=analisis, x="energia_wh", y="exactitud_pct", hue="configuracion", style="configuracion", palette=paleta, s=80, ax=ax)
ax.set(title="Exactitud y energía en observaciones simuladas", xlabel="Energía (Wh)", ylabel="Exactitud (%)")
ax.legend(title="Configuración")
ax.annotate("Dispositivo edge", xy=(12.4, 70), xytext=(15.5, 69), arrowprops={"arrowstyle": "->"})
fig.tight_layout()
plt.show()


## 4. Exportar sin rutas absolutas

Una ruta absoluta solo funciona en el equipo de quien la escribió. Usa `Path` y una carpeta relativa al curso. La celda siguiente está comentada para no generar archivos al ejecutar el cuaderno de ejemplo. Al entregar una actividad, comprueba que la figura se exportó y que el archivo generado no se añade al repositorio salvo que se solicite.


In [ ]:
# salida = Path("unidades") / "salidas" / "45_exactitud_energia.png"
# salida.parent.mkdir(parents=True, exist_ok=True)
# fig.savefig(salida, dpi=150, bbox_inches="tight")


## Práctica guiada

Filtra solo `dispositivo == "servidor"`, crea una columna `latencia_por_mb` mediante una operación vectorizada y construye una dispersión de memoria frente a latencia con configuración como color y estilo. Indica en una frase que la figura describe únicamente ese subconjunto.


In [ ]:
# Completa la práctica guiada aquí.


## Ejercicios

### Ejercicio 45.01 · Transformación trazable

Crea una copia llamada `edge` con las observaciones del dispositivo edge. Añade `eficiencia_pct_wh` y verifica que tiene 12 filas y no contiene valores ausentes.

### Ejercicio 45.02 · Resumen y figura coherentes

Agrupa por configuración y calcula medias de exactitud y energía. Representa esas medias con un gráfico que indique explícitamente que son medias simuladas.

### Ejercicio 45.03 · Pivote y mapa de calor

Crea una tabla pivote de latencia media por configuración y dispositivo. Represéntala con un `heatmap` con anotaciones, barra de color y unidades.

### Ejercicio 45.04 · Diseño accesible

Crea una dispersión de latencia frente a exactitud. Usa color y estilo para configuración, título, etiquetas y leyenda. Escribe por qué una persona no debe depender solo del color para interpretar grupos.

### Ejercicio 45.05 · Miniproyecto: mini EDA reproducible

Entrega una secuencia de cuatro pasos: inspección de calidad, transformación documentada, una tabla de resumen y dos figuras complementarias. Una figura debe mostrar puntos individuales; la otra puede resumir con paneles o un mapa de calor. Redacta conclusiones, límites y la ruta relativa que usarías para exportar una figura.


In [ ]:
# Escribe aquí tu solución de los ejercicios 45.01 a 45.05.


## Autoevaluación y cierre

1. ¿Por qué separar el `DataFrame` original del transformado?
2. ¿Qué debe indicar el título de una figura basada en medias?
3. ¿Qué información debe conservar una figura accesible además del color?
4. ¿Por qué exportar con una ruta relativa?
5. ¿Qué componentes hacen reproducible un EDA?

Has completado la preparación para el proyecto 46, donde aplicarás el flujo completo sobre un conjunto de datos documentado.
